# Producion Agricola (Analisis Exploratorio de Datos)

Una vez realizados los ajustes pertinentes al conjunto de datos principal. Procedemos a realizar el analisis exploratorio de datos.
Donde conoceremos los datos y los iremos preparando para los futuroes modelos planteados.

**Observación** Una de las cosas distintivas con respecto a las etapas pasadas es que si nuestro problema lo requiere conectaremos nuestros datos principales con otras fuentes. Como veremos más adelante

In [78]:
from scripts.config import PRELOADS, DATA_DIR
import os
import pickle
import pandas as pd

# Notemos que ahora estamos obteniendo la información ya normalizada.
with open(PRELOADS['AGROPROD_2'], "rb") as preprocess: 
    dgsiap = pickle.load(preprocess)

dgsiap

Comenzamos con la carga de los conjuntos de datos de Producción Agricola de parte del DGSIAP.

In [79]:
dataset = [file.open(pars=[1, {"low_memory":False, "index_col":0}]) for file in dgsiap.branch]

In [112]:
q = dgsiap.root
q.open()['Observaciones'][20]

'Volumen de producción de la superficie cosechada cuya unidad de medida son las toneladas, con excepción del maguey pulquero y trigo ornamental, con una métrica de miles de litros y gruesas, respectivamente.'

Recordemos que en esta primera parte procuramos estimar la tasa de éxito de un cultivo en función del clima de la región. Evitando confundir con el rendimiento. La tasa de exito en un cultivo es el cociente entre las hectareas cosechadas y las hectarias sembradas. Para este analisis omitiremos las hectareas siniestradas.

Primero haremos conjunto de datos que considere todos los años.

In [80]:
prodagro = dataset[0]

Como EDA de muestra usaremos un cultivo arbitrario.

In [81]:
# Buscamos todos los cultivos.
crops = prodagro['Nomcultivo'].unique().tolist()
crop_s = crops[20]
crop_s

'Uva'

Tomando la Uva como ejemplo. Vemos los estados que la cosechan 

In [82]:
prod_crops = prodagro[prodagro['Nomcultivo'] == crop_s]
prod_crops['Nomestado'].unique().tolist()

['Aguascalientes',
 'Baja California',
 'Baja California Sur',
 'Coahuila',
 'Chihuahua',
 'Durango',
 'Guanajuato',
 'Jalisco',
 'Nayarit',
 'Nuevo León',
 'Puebla',
 'Querétaro',
 'San Luis Potosí',
 'Sinaloa',
 'Sonora',
 'Zacatecas']

Tambien veremos los ciclos de cosecha.

In [83]:
prod_crops['Nomcicloproductivo'].unique()

<StringArray>
['Perennes']
Length: 1, dtype: str

En este caso tenemos perenne. Es decir, la Uva tarda más de un año en cosecharse. Por lo tanto, el volumen de producción puede ser de varios años.

In [84]:
prod_crops['Sembrada'].sum(), prod_crops['Cosechada'].sum()

(np.float64(38064.91), np.float64(37714.33))

Teniendo que en 2025 Hubo 38064.91 ha Sembradas y 37714.33 ha Cosechadas. 

In [85]:
f"Tasa de Exito Nacional: {prod_crops['Cosechada'].sum() / prod_crops['Sembrada'].sum() * 100}"

'Tasa de Exito Nacional: 99.07899427583041'

La uva tiene una tasa de exito en Mexico de aproximadamente 99.079%. Haremos lo mismo con por estado.

In [86]:
prod_crop_ent = prod_crops.groupby("Nomestado")[['Sembrada','Cosechada']].sum()
prod_crop_ent['TasaExito'] = prod_crop_ent['Cosechada'] / prod_crop_ent['Sembrada']
prod_crop_ent

,Sembrada,Cosechada,TasaExito
Nomestado,,,
Aguascalientes,1167.04,1102.04,0.944304
Baja California,4493.49,4488.49,0.998887
Baja California Sur,48.50,10.50,0.216495
Chihuahua,422.83,294.00,0.695315
Coahuila,767.50,676.50,0.881433
Durango,3.00,3.00,1.000000
Guanajuato,511.00,509.00,0.996086
Jalisco,1458.50,1458.50,1.000000
Nayarit,108.00,107.50,0.995370


Si vemos la Tasa de Éxito Nacional como una suma ponderada de los estados

$$
    r = \sum_{e\in E} \alpha_e r_e, \sum_{e\in E}\alpha_e = 1.
$$

Podemos deducir que 

$$
    \alpha_e = \dfrac{s_e}{s}, s = \sum_{e \in E}s_e
$$

In [87]:
prod_crop_ent['Pond'] = prod_crop_ent['Sembrada'] / prod_crop_ent['Sembrada'].sum()
print("Error", (prod_crop_ent['TasaExito'] * prod_crop_ent['Pond']).sum()-prod_crops['Cosechada'].sum() / prod_crops['Sembrada'].sum())
prod_crop_ent

Error -1.1102230246251565e-16


,Sembrada,Cosechada,TasaExito,Pond
Nomestado,,,,
Aguascalientes,1167.04,1102.04,0.944304,0.030659
Baja California,4493.49,4488.49,0.998887,0.118048
Baja California Sur,48.50,10.50,0.216495,0.001274
Chihuahua,422.83,294.00,0.695315,0.011108
Coahuila,767.50,676.50,0.881433,0.020163
Durango,3.00,3.00,1.000000,0.000079
Guanajuato,511.00,509.00,0.996086,0.013424
Jalisco,1458.50,1458.50,1.000000,0.038316
Nayarit,108.00,107.50,0.995370,0.002837


In [88]:
top_c_ent = prod_crop_ent.sort_values('Pond', ascending=False)
print("Aporte TOP 3", top_c_ent.head(n=3)['Pond'].sum())
top_c_ent

Aporte TOP 3 0.862040393632876


,Sembrada,Cosechada,TasaExito,Pond
Nomestado,,,,
Sonora,20388.00,20388.00,1.000000,0.535611
Zacatecas,7932.00,7932.00,1.000000,0.208381
Baja California,4493.49,4488.49,0.998887,0.118048
Jalisco,1458.50,1458.50,1.000000,0.038316
Aguascalientes,1167.04,1102.04,0.944304,0.030659
Coahuila,767.50,676.50,0.881433,0.020163
Querétaro,530.00,529.75,0.999528,0.013924
Guanajuato,511.00,509.00,0.996086,0.013424
Chihuahua,422.83,294.00,0.695315,0.011108


Vemos que más del 85% de la tasa de exito nacional depende de 3 estados (Sonora, Zacatecas y Baja California). Ahora veremos lo mismo pero con el volumen de producción.


In [89]:
vprod_crop_ent = prod_crops.groupby("Nomestado")[['Sembrada','Cosechada','Volumenproduccion']].sum()
vprod_crop_ent['Pond'] = vprod_crop_ent['Volumenproduccion'] / vprod_crop_ent['Volumenproduccion'].sum()
vprod_crop_ent.sort_values('Volumenproduccion', ascending=False, inplace=True)
print(vprod_crop_ent.sort_values('Volumenproduccion', ascending=False).head(n=3)['Pond'].sum())
vprod_crop_ent

0.8914637667354417


,Sembrada,Cosechada,Volumenproduccion,Pond
Nomestado,,,,
Sonora,20388.00,20388.00,340983.18,0.646887
Zacatecas,7932.00,7932.00,93896.38,0.178133
Baja California,4493.49,4488.49,35023.57,0.066444
Jalisco,1458.50,1458.50,24790.98,0.047032
Aguascalientes,1167.04,1102.04,13843.83,0.026263
Coahuila,767.50,676.50,4970.05,0.009429
Guanajuato,511.00,509.00,4630.86,0.008785
Querétaro,530.00,529.75,4382.05,0.008313
Nayarit,108.00,107.50,1711.35,0.003247


En este caso se mantuvo el orden de los estados que más producen. Además que el 89% de la producción nacional depende de los mismos estados. Ahora veremos el valor de la Uva por estado.

In [120]:
prod_crop_ent = prod_crops.groupby("Nomestado")[['Sembrada','Cosechada','Volumenproduccion','Valorproduccion']].sum()
prod_crop_ent['PrecioPond'] = prod_crop_ent['Valorproduccion'] / prod_crop_ent['Volumenproduccion']
prod_crop_ent['TasaExito'] = prod_crop_ent['Cosechada'] / prod_crop_ent['Sembrada']
prod_crop_ent.sort_values("PrecioPond", ascending=False)

,Sembrada,Cosechada,Volumenproduccion,Valorproduccion,PrecioPond,TasaExito
Nomestado,,,,,,
Sonora,20388.00,20388.00,340983.18,1.103440e+10,32360.545504,1.000000
Puebla,101.75,81.75,485.20,1.324992e+07,27308.168075,0.803440
Jalisco,1458.50,1458.50,24790.98,5.765078e+08,23254.738719,1.000000
Baja California,4493.49,4488.49,35023.57,8.141711e+08,23246.377001,0.998887
Chihuahua,422.83,294.00,1403.10,2.482460e+07,17692.680073,0.695315
Baja California Sur,48.50,10.50,36.76,6.049000e+05,16455.386289,0.216495
Coahuila,767.50,676.50,4970.05,7.781708e+07,15657.203366,0.881433
San Luis Potosí,126.00,126.00,905.70,1.343080e+07,14829.192889,1.000000
Querétaro,530.00,529.75,4382.05,6.124140e+07,13975.514538,0.999528


Notemos que Sonora mantiene el primer lugar en precio de venta con $32,360.55 MXN por tonelada de Uva. Sin embargo, los otros dos con la uva más cara son Puebla y Jalisco. Entonces la Uva producida en estos estados es más valiosa que la Uva producida en Zacatecas y Baja California. Entonces podriamos ver si aumentando las hectareas sembradas, manteniendo sus tasas de éxito podemos aumentar la ganancia nacional de la Uva. 

De esto vemos que Sonora, Zacatecas y Baja California representan más del 85% del éxito de los cultivos de Uva. Ahora veamos esto para el resto de cultivos.

In [91]:
def sucess_rate(crop, df):
    prod_crop = df[df['Nomcultivo']==crop]
    crop_rate = prod_crop['Cosechada'].sum() / prod_crop['Sembrada'].sum()
    return crop_rate


all_rate_crops = pd.DataFrame(index=crops, columns=['TasaExitoNacional'])
for crop in all_rate_crops.index:
    all_rate_crops.loc[crop, 'TasaExitoNacional'] = sucess_rate(crop, prodagro)


/tmp/ipykernel_101034/2263796466.py:3: RuntimeWarning: invalid value encountered in scalar divide
  crop_rate = prod_crop['Cosechada'].sum() / prod_crop['Sembrada'].sum()
/tmp/ipykernel_101034/2263796466.py:3: RuntimeWarning: invalid value encountered in scalar divide
  crop_rate = prod_crop['Cosechada'].sum() / prod_crop['Sembrada'].sum()


In [92]:
all_rate_crops[all_rate_crops['TasaExitoNacional']<0.5]
print(len(all_rate_crops[all_rate_crops['TasaExitoNacional']<0.5]) / len(crops) * 100)

2.6058631921824107


Vemos que durante 2025. Solo 2.6% de cultivos tuvieron menos del 50% de efectividad. Aplicando esto a todos los años veremos cuales han sido los mejores cultivos en México.

In [93]:
crop_history= pd.DataFrame(index=crops, columns=[f'{y}' for y in range(2003, 2026)])
for k, prodagro in enumerate(dataset):
    year = f"{2025-k}"
    for crop in crop_history.index:
        crop_history.loc[crop, f'{year}'] = sucess_rate(crop, prodagro)

crop_history

/tmp/ipykernel_101034/2263796466.py:3: RuntimeWarning: invalid value encountered in scalar divide
  crop_rate = prod_crop['Cosechada'].sum() / prod_crop['Sembrada'].sum()
/tmp/ipykernel_101034/2263796466.py:3: RuntimeWarning: invalid value encountered in scalar divide
  crop_rate = prod_crop['Cosechada'].sum() / prod_crop['Sembrada'].sum()
/tmp/ipykernel_101034/2263796466.py:3: RuntimeWarning: invalid value encountered in scalar divide
  crop_rate = prod_crop['Cosechada'].sum() / prod_crop['Sembrada'].sum()
/tmp/ipykernel_101034/2263796466.py:3: RuntimeWarning: invalid value encountered in scalar divide
  crop_rate = prod_crop['Cosechada'].sum() / prod_crop['Sembrada'].sum()
/tmp/ipykernel_101034/2263796466.py:3: RuntimeWarning: invalid value encountered in scalar divide
  crop_rate = prod_crop['Cosechada'].sum() / prod_crop['Sembrada'].sum()
/tmp/ipykernel_101034/2263796466.py:3: RuntimeWarning: invalid value encountered in scalar divide
  crop_rate = prod_crop['Cosechada'].sum() / pr

KeyboardInterrupt: 

In [ ]:
principal_crops = crop_history.isna().sum(axis=1).sort_values(ascending=False)
x = principal_crops[principal_crops==0].index.tolist()
x

In [ ]:
crop_history.loc[x]

In [ ]:
import matplotlib.pyplot as plt

plt.plot(crop_history.columns, crop_history.loc['Nopalitos'])
plt.ylim([0,1])